<a href="https://colab.research.google.com/github/Chandanrohit/crops_disease_detector/blob/main/demo_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os
from pathlib import Path

def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

pip("streamlit", "pyngrok", "onnxruntime", "Pillow", "numpy")
print("✅ Packages ready")

# ── Verify model files exist in Drive ─────────────────────────────────────────
DRIVE = "/content/drive/MyDrive/Fasal_AI"
ONNX  = f"{DRIVE}/checkpoints/crop_disease_model_int8.onnx"
LBLS  = f"{DRIVE}/checkpoints/class_labels.json"

missing = [p for p in [ONNX, LBLS] if not Path(p).exists()]
if missing:
    print("❌ Missing files in Drive:")
    for m in missing: print(f"   {m}")
    print("\n   These are created by the full training notebook.")
    print("   Run Section 4 (Training) + Section 6 (Export) first.")
else:
    size = os.path.getsize(ONNX) / 1e6
    import json
    classes = json.load(open(LBLS))
    print(f"✅ Model found: {size:.1f} MB — {len(classes)} disease classes")
    print("✅ Labels found")
    print("\n👉 Run Cell 2 to clone your repo and get the latest demo.py")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Packages ready
❌ Missing files in Drive:
   /content/drive/MyDrive/Fasal_AI/checkpoints/crop_disease_model_int8.onnx
   /content/drive/MyDrive/Fasal_AI/checkpoints/class_labels.json

   These are created by the full training notebook.
   Run Section 4 (Training) + Section 6 (Export) first.


In [3]:
import subprocess, shutil, os
from pathlib import Path

# ── ✏️  YOUR REPO URL ─────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/Chandanrohit/crops_disease_detector.git"
REPO_DIR  = "/content/fasal_repo"
DRIVE     = "/content/drive/MyDrive/FasalAI"
# ──────────────────────────────────────────────────────────────────────────────

# Clone or pull latest code
if Path(REPO_DIR).exists():
    subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True)
    print("✅ Repo updated (git pull)")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True)
    print("✅ Repo cloned")

# Copy model files from Drive into the repo's checkpoints folder
# (demo.py reads from ./checkpoints/ relative to where streamlit runs)
ckpt_dir = f"{REPO_DIR}/checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)

files_to_copy = {
    f"{DRIVE}/checkpoints/crop_disease_model_int8.onnx": f"{ckpt_dir}/crop_disease_model_int8.onnx",
    f"{DRIVE}/checkpoints/crop_disease_model.onnx":      f"{ckpt_dir}/crop_disease_model.onnx",
    f"{DRIVE}/checkpoints/class_labels.json":            f"{ckpt_dir}/class_labels.json",
}

for src, dst in files_to_copy.items():
    if Path(src).exists():
        shutil.copy2(src, dst)
        mb = os.path.getsize(dst) / 1e6
        print(f"  ✅ {Path(dst).name}  ({mb:.1f} MB)")
    else:
        print(f"  ⚠️  Skipped (not found): {Path(src).name}")

# Confirm demo.py exists
demo_py = Path(f"{REPO_DIR}/demo.py")
if demo_py.exists():
    print(f"\n✅ demo.py found in repo")
else:
    print(f"\n⚠️  demo.py not found — check your repo at {REPO_URL}")

print("\n👉 Run Cell 3 to launch the demo and get your public URL")




✅ Repo cloned
  ✅ crop_disease_model_int8.onnx  (6.1 MB)
  ✅ crop_disease_model.onnx  (19.6 MB)
  ✅ class_labels.json  (0.0 MB)

⚠️  demo.py not found — check your repo at https://github.com/Chandanrohit/crops_disease_detector.git

👉 Run Cell 3 to launch the demo and get your public URL


In [6]:
NGROK_TOKEN = "3DSkzK8QDyf3w2pOpXifNfyyqU3_5T2cpuzMbomcuxJXmgqy6"
import threading, time, subprocess, os
from pyngrok import ngrok, conf

REPO_DIR = "/content/fasal_repo"   # must match Cell 2

# Kill any stale processes from previous runs
os.system("pkill -f 'streamlit run' 2>/dev/null")
ngrok.kill()
time.sleep(1.5)

conf.get_default().auth_token = NGROK_TOKEN
PORT = 8501

def _run():
    subprocess.run([
        "streamlit", "run", f"{REPO_DIR}/demo.py",
        "--server.port",               str(PORT),
        "--server.headless",           "true",
        "--server.enableCORS",         "false",
        "--server.enableXsrfProtection", "false",
        "--browser.gatherUsageStats",  "false",
    ], cwd=REPO_DIR)

threading.Thread(target=_run, daemon=True).start()
print("⏳ Starting…")
time.sleep(5)

url = ngrok.connect(PORT, "http").public_url

print(f"""
╔══════════════════════════════════════════════════╗
║  🌿  FASAL AI IS LIVE                           ║
╠══════════════════════════════════════════════════╣
║  {url:<48} ║
╚══════════════════════════════════════════════════╝
""")

⏳ Starting…

╔══════════════════════════════════════════════════╗
║  🌿  FASAL AI IS LIVE                           ║
╠══════════════════════════════════════════════════╣
║  https://certified-overdrive-vanish.ngrok-free.dev ║
╚══════════════════════════════════════════════════╝

